# Task 1: Model Training and Optimization Pipeline
Use this notebook to perform your data preprocessing, hyperparameter tuning via Cross-Validation, and final evaluation on the test set.

In [ ]:
import os
from google.colab import files
import shutil

# create Dataset folder
os.makedirs("Dataset", exist_ok=True)

print("Upload train.csv")
uploaded = files.upload()
for f in uploaded.keys():
    shutil.move(f, "Dataset/train.csv")

print("Upload test.csv")
uploaded = files.upload()
for f in uploaded.keys():
    shutil.move(f, "Dataset/test.csv")

print("Upload requirements.txt")
uploaded = files.upload()
# stays in root

print("Setup complete")
print("Dataset contents:", os.listdir("Dataset"))

Upload train.csv


Upload test.csv


Upload requirements.txt


Setup complete
Dataset contents: ['test.csv', 'README.md', 'train.csv']


In [ ]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.9 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    F

In [ ]:
!pip install optuna
!pip install trackio
!pip install kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import pickle
import time
import optuna
import trackio
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Add any other imports you need here
import os
import optuna.visualization as vis
import plotly.io as pio

## 1. Data Loading & Preprocessing
Load `train.csv` and `test.csv`. Convert string categorical variables to numeric.
**Required:** Save your label encoders/mappings because your Streamlit UI will need them later to prepare user inputs for inference!

In [ ]:
train_df = pd.read_csv('Dataset/train.csv')
test_df = pd.read_csv('Dataset/test.csv')
# Remove exact duplicate rows
print("Duplicate rows in train:", train_df.duplicated().sum())
train_df = train_df.drop_duplicates()

# Handling missing values (LabelEncoder cannot handle NaN)
print(train_df.isnull().sum())
print(test_df.isnull().sum())
train_df = train_df.fillna("missing")
test_df = test_df.fillna("missing")

# TODO: Implement your preprocessing here (use LabelEncoder or manual dictionaries)
encoders = {}

for col in train_df.select_dtypes(include='object').columns:
    le = LabelEncoder()

    # Combining train + test to avoid unseen category issues
    combined = pd.concat([train_df[col], test_df[col]], axis=0)
    le.fit(combined)

    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

    encoders[col] = le

# TODO: Separate predictors (X) and target (y: 'price')
X_train = train_df.drop('price', axis=1)
y_train = train_df['price']
if 'price' in test_df.columns:
    X_test = test_df.drop('price', axis=1)
    y_test = test_df['price']
else:
    X_test = test_df
    y_test = None


# save encoders (required for UI)
os.makedirs("models", exist_ok=True)

with open("models/encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

Duplicate rows in train: 1439
location             0
city                 0
latitude             0
longitude            0
price                0
numBathrooms         0
numBalconies         0
isNegotiable         0
SecurityDeposit      0
Status               0
Size_ft²             0
BHK                  0
rooms_num            0
property_type        0
verification_days    0
dtype: int64
location             0
city                 0
latitude             0
longitude            0
price                0
numBathrooms         0
numBalconies         0
isNegotiable         0
SecurityDeposit      0
Status               0
Size_ft²             0
BHK                  0
rooms_num            0
property_type        0
verification_days    0
dtype: int64


## 2. Hyperparameter Tuning using Cross-Validation

**Strict Search Space:**
- `n_estimators`: 50 to 200
- `max_depth`: 10 to 30
- `min_samples_split`: 2 to 10

Implement Grid Search, Random Search, and Bayesian Optimization (using Optuna). Evaluate each using 5-fold cross-validation on `train_df`.

In [ ]:
from scipy.stats import randint

rf = RandomForestRegressor(random_state=42)
# TODO: Initialize trackio project/experiment here
# Initialize trackio
trackio.init(
    project="assignment4",
    name="rf_hyperparameter_tuning"
)

trackio.log({
    "model_type": "RandomForestRegressor",
    "random_state": 42
})

grid_space = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [10, 15, 20, 25, 30],
    "min_samples_split": [2, 5, 8]
}

random_space = {
    "n_estimators": randint(50, 201),
    "max_depth": randint(10, 31),
    "min_samples_split": randint(2, 11)
}
# TODO: 1. Grid Search Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score

# 1. GRID SEARCH

trackio.log({"experiment": "grid_search", "method": "grid"})
start = time.time()

grid = GridSearchCV(
    rf,
    param_grid=grid_space,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

grid_time = time.time() - start
grid_best_score = -grid.best_score_

# Trial-wise logging
for i, params in enumerate(grid.cv_results_['params']):
    trackio.log({
        "method": "grid",
        "trial": i,
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "min_samples_split": params["min_samples_split"],
        "mae": float(-grid.cv_results_['mean_test_score'][i])
    })

# Summary log
trackio.log({
    "grid_time": float(grid_time),
    "grid_iterations": int(len(grid.cv_results_["params"])),
    "grid_best_score": float(grid_best_score)
})
trackio.log({
    "grid_best_params": grid.best_params_
})
# TODO: 2. Random Search Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score

# 2. RANDOM SEARCH

trackio.log({"experiment": "random_search", "method": "random"})
start = time.time()

random = RandomizedSearchCV(
    rf,
    param_distributions=random_space,
    n_iter=60,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1
)

random.fit(X_train, y_train)

random_time = time.time() - start
random_best_score = -random.best_score_

#  Trial-wise logging
for i, params in enumerate(random.cv_results_['params']):
    trackio.log({
        "method": "random",
        "trial": i,
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "min_samples_split": params["min_samples_split"],
        "mae": float(-random.cv_results_['mean_test_score'][i])
    })

# Summary log
trackio.log({
    "random_time": float(random_time),
    "random_iterations": int(60),
    "random_best_score": float(random_best_score)
})
trackio.log({
    "random_best_params": random.best_params_
})
# TODO: 3. Bayesian Optimization (Optuna) Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score

# 3. OPTUNA (BAYESIAN)

trackio.log({"experiment": "optuna_search", "method": "optuna"})
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 10, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42,
        n_jobs=-1
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=1
    ).mean()

    mae = -score

    # Trial-wise logging
    trackio.log({
        "method": "optuna",
        "trial": trial.number,
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "min_samples_split": min_samples_split,
        "mae": float(mae)
    })

    return mae

start = time.time()

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=60)

optuna_time = time.time() - start
optuna_best_score = study.best_value

# Summary log
trackio.log({
    "optuna_time": float(optuna_time),
    "optuna_iterations": int(60),
    "optuna_best_score": float(optuna_best_score)
})
trackio.log({
    "optuna_best_params": study.best_params
})

#  FINAL COMPARISON

print("Grid Best:", grid.best_params_, grid_best_score)
print("Random Best:", random.best_params_, random_best_score)
print("Optuna Best:", study.best_params, optuna_best_score)

best_score = min(grid_best_score, random_best_score, optuna_best_score)

if best_score == grid_best_score:
    best_model = grid.best_estimator_
    best_method = "grid"
    print("Best method is grid")
elif best_score == random_best_score:
    best_model = random.best_estimator_
    best_method = "random"
    print("Best method is random")
else:
    best_model = RandomForestRegressor(**study.best_params, random_state=42, n_jobs=-1)
    best_method = "optuna"
    print("Best method is optuna")

best_model.fit(X_train, y_train)

# Final log
trackio.log({
    "final_grid_mae": float(grid_best_score),
    "final_random_mae": float(random_best_score),
    "final_optuna_mae": float(optuna_best_score),
    "best_method": best_method,
    "grid_time": float(grid_time),
    "random_time": float(random_time),
    "optuna_time": float(optuna_time)
})

trackio.finish()

* Trackio project initialized: assignment4
* Trackio metrics logged to: /root/.cache/huggingface/trackio


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


* Created new run: rf_hyperparameter_tuning
* Running on public URL: https://926b5b91b57e5ca614.gradio.live


[I 2026-04-14 18:47:51,561] A new study created in memory with name: no-name-cf299db1-a249-4651-ae8f-fcb7d5565f8d
[I 2026-04-14 18:48:14,557] Trial 0 finished with value: 15586.127148041429 and parameters: {'n_estimators': 106, 'max_depth': 29, 'min_samples_split': 8}. Best is trial 0 with value: 15586.127148041429.
[I 2026-04-14 18:48:43,653] Trial 1 finished with value: 15752.662280799379 and parameters: {'n_estimators': 140, 'max_depth': 13, 'min_samples_split': 3}. Best is trial 0 with value: 15586.127148041429.
[I 2026-04-14 18:48:58,504] Trial 2 finished with value: 15646.17867206714 and parameters: {'n_estimators': 58, 'max_depth': 28, 'min_samples_split': 7}. Best is trial 0 with value: 15586.127148041429.
[I 2026-04-14 18:49:23,383] Trial 3 finished with value: 16702.54940451445 and parameters: {'n_estimators': 156, 'max_depth': 10, 'min_samples_split': 10}. Best is trial 0 with value: 15586.127148041429.
[I 2026-04-14 18:50:03,678] Trial 4 finished with value: 15664.262785641

Grid Best: {'max_depth': 25, 'min_samples_split': 5, 'n_estimators': 200} 15479.407834646909
Random Best: {'max_depth': 30, 'min_samples_split': 3, 'n_estimators': 160} 15465.766898274067
Optuna Best: {'n_estimators': 177, 'max_depth': 25, 'min_samples_split': 3} 15448.091240197336
Best method is optuna
* Run finished. Uploading logs to Trackio (please wait...)


## 3. Evaluation & Plots
Plot the compute trials (iterations) vs. cross-validation error, and plot the hyperparameter space to visualize how the Bayesian method explored the space.

In [ ]:
# IMPORTS
os.makedirs("plots", exist_ok=True)

# OPTUNA ERRORS (BEST SO FAR)
optuna_errors = [trial.value for trial in study.trials if trial.value is not None]

optuna_best = []
best = float("inf")

for val in optuna_errors:
    best = min(best, val)
    optuna_best.append(best)

# GRID SEARCH ERRORS
grid_errors = [-score for score in grid.cv_results_['mean_test_score']]

grid_best = []
best = float("inf")

for val in grid_errors:
    best = min(best, val)
    grid_best.append(best)

# RANDOM SEARCH ERRORS
random_errors = [-score for score in random.cv_results_['mean_test_score']]

random_best = []
best = float("inf")

for val in random_errors:
    best = min(best, val)
    random_best.append(best)

# PLOT 1: TRIALS VS ERROR

plt.plot(range(1, len(grid_best)+1), grid_best, label="Grid Search")
plt.plot(range(1, len(random_best)+1), random_best, label="Random Search")
plt.plot(range(1, len(optuna_best)+1), optuna_best, label="Bayesian (Optuna)")
plt.xlabel("Number of Trials / Iterations")
plt.ylabel("Best Mean CV MAE Found So Far")
plt.title("Trials vs Error Comparison")

plt.legend()
plt.grid(True)

plt.savefig("plots/trials_vs_error.png")
plt.show()
plt.close()

# PLOT 2: OPTUNA HYPERPARAMETER SPACE (CUSTOM)

import matplotlib.pyplot as plt

# Extract trial numbers and values
trials = []
errors = []

for trial in study.trials:
    if trial.value is not None:
        trials.append(trial.number)
        errors.append(trial.value)

# Compute best-so-far curve
best_values = []
best = float("inf")

for val in errors:
    best = min(best, val)
    best_values.append(best)

# Plot
plt.figure(figsize=(10, 6))

plt.plot(trials, errors, label="Trial MAE", linestyle="--")
plt.plot(trials, best_values, label="Best MAE So Far", linewidth=2)

plt.xlabel("Trial Number")
plt.ylabel("MAE")
plt.title("Optuna Optimization History")

plt.legend()
plt.grid(True)

plt.savefig("plots/optuna_hyperparameter_space.png")
plt.show()
plt.close()

## 4. Final Testing & Model Saving
Report the best hyperparameters found, train your overall best model on the entire `train.csv`, evaluate on `test.csv`, and save the model file.

In [ ]:
# TODO: Print the best hyperparameters found by all 3 methods
print("Grid Search Best Params:", grid.best_params_)
print("Grid Search Best MAE:", float(grid_best_score))

print("Random Search Best Params:", random.best_params_)
print("Random Search Best MAE:", float(random_best_score))

print("Optuna Best Params:", study.best_params)
optuna_best_mae = study.best_value  # FIXED
print("Optuna Best MAE:", float(optuna_best_mae))

print("\n=== Cross-Validation Performance Comparison ===")
print(f"Grid Search CV MAE: {grid_best_score:.4f}")
print(f"Random Search CV MAE: {random_best_score:.4f}")
print(f"Optuna CV MAE: {optuna_best_mae:.4f}")

# TODO: Train the best model found on the full X_train
best_score = min(grid_best_score, random_best_score, optuna_best_mae)  # FIXED

if best_score == grid_best_score:
    print("\nBest Method: Grid Search")
    best_model = grid.best_estimator_

elif best_score == random_best_score:
    print("\nBest Method: Random Search")
    best_model = random.best_estimator_

else:
    print("\nBest Method: Bayesian Optimization (Optuna)")
    best_model = RandomForestRegressor(**study.best_params, random_state=42, n_jobs=-1)

best_model.fit(X_train, y_train)

# TODO: Evaluate the model on X_test (Report MAE)

y_pred = best_model.predict(X_test)
if y_test is not None:
    test_mae = mean_absolute_error(y_test, y_pred)
    print("\nFinal Test MAE:", float(test_mae))
else:
    print("\nTest MAE cannot be computed (no target in test set)")

# TODO: Save best_model.pkl and any necessary encoders to the models/ folder
os.makedirs("models", exist_ok=True)

with open("models/best_rf_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("models/encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)
with open("models/feature_columns.pkl", "wb") as f:
    pickle.dump(list(X_train.columns), f)

print("\nModel and encoders saved successfully!")

Grid Search Best Params: {'max_depth': 25, 'min_samples_split': 5, 'n_estimators': 200}
Grid Search Best MAE: 15479.407834646909
Random Search Best Params: {'max_depth': 30, 'min_samples_split': 3, 'n_estimators': 160}
Random Search Best MAE: 15465.766898274067
Optuna Best Params: {'n_estimators': 177, 'max_depth': 25, 'min_samples_split': 3}
Optuna Best MAE: 15448.091240197336

=== Cross-Validation Performance Comparison ===
Grid Search CV MAE: 15479.4078
Random Search CV MAE: 15465.7669
Optuna CV MAE: 15448.0912

Best Method: Bayesian Optimization (Optuna)

Final Test MAE: 12585.100045709018

Model and encoders saved successfully!


In [ ]:
from google.colab import files

files.download("models/best_rf_model.pkl")
files.download("models/encoders.pkl")
files.download("plots/trials_vs_error.png")
files.download("plots/optuna_hyperparameter_space.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>